# DarkSight — E4: train the detector ON brightened photos

E3 showed what happens when you brighten photos *at test time only*: the detector was trained on dark
photos, so brightened ones look unfamiliar to it. **E4 removes that excuse** — here the detector is
trained on brightened photos and tested on brightened photos, so both sides match.

**E2 vs E4 is the main result of the whole project.**

| | trained on | tested on |
|---|---|---|
| E2 | raw dark | raw dark |
| E3 | raw dark | brightened |
| **E4** | **brightened** | **brightened** |

### Settings
**Accelerator: GPU T4 x2** · **Internet: On** · **Persistence: Files and Variables** → **Run All** (~2–2.5 h).

### Optional but recommended
Attach the output of notebook 05 (**Add Input → Your Work → Notebook Output**). If `det_raw.pt` is found,
this notebook also re-runs E2 and E3 so the final table has all three side by side.

## Step 0 · Setup

In [ ]:
!pip -q install ultralytics gdown

import os, shutil, time, json
from pathlib import Path
from collections import Counter
import numpy as np, torch, cv2, pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

START = time.time()
WORK, TEMP = Path('/kaggle/working'), Path('/kaggle/temp')
DEVICE = [0, 1]            # both T4s. Change to 0 if multi-GPU training misbehaves.
MAX_SIDE = 1280

CLASSES = ["Bicycle", "Boat", "Bottle", "Bus", "Car", "Cat",
           "Chair", "Cup", "Dog", "Motorbike", "People", "Table"]
CLASS_ID = {n.lower(): i for i, n in enumerate(CLASSES)}
print('GPUs:', torch.cuda.device_count())

In [ ]:
# Two small files from our repo: the official split list and the Zero-DCE weights.
!git clone -q --depth 1 https://github.com/Manideep-21/darksight.git {TEMP}/repo
SPLIT_FILE, ZERODCE_FILE = TEMP/'repo/configs/exdark_imageclasslist.txt', TEMP/'repo/weights/zerodce_epoch99.pth'

# ExDark itself (1.5 GB photos + 5 MB boxes) from the authors' Google Drive.
SRC = TEMP / 'exdark_src'
if not SRC.exists():
    !gdown -q 1BHmPgu8EsHoFDDkMGLVoXIlCth2dW6Yx -O {TEMP}/images.zip
    !gdown -q 1P3iO3UYn7KoBi5jiUkogJq96N6maZS1i -O {TEMP}/boxes.zip
    !mkdir -p {SRC} && unzip -q -o {TEMP}/images.zip -d {SRC} && unzip -q -o {TEMP}/boxes.zip -d {SRC}
    !rm -rf {SRC}/__MACOSX

photos = {p.stem.lower(): p for p in (SRC/'ExDark').rglob('*') if p.is_file() and not p.name.startswith('.')}
boxes  = {p.name.lower().removesuffix('.txt').rsplit('.', 1)[0]: p for p in (SRC/'ExDark_Annno').rglob('*.txt')}

SPLIT_NAME = {'1': 'train', '2': 'val', '3': 'test'}
LIGHTING = {'1':'Low','2':'Ambient','3':'Object','4':'Single','5':'Weak',
            '6':'Strong','7':'Screen','8':'Window','9':'Shadow','10':'Twilight'}
split_of, lighting_of = {}, {}
for line in SPLIT_FILE.read_text().splitlines():
    c = line.split()
    if len(c) == 5 and c[1].isdigit():
        k = Path(c[0]).stem.lower()
        split_of[k], lighting_of[k] = SPLIT_NAME[c[4]], LIGHTING[c[2]]
print(len(photos), 'photos |', Counter(split_of.values()))

## Step 1 · Convert ExDark to YOLO format (~8 min)

In [ ]:
def convert_one(photo_path, box_path, out_photo, out_label):
    """Save one photo as PNG and its boxes as a YOLO label file (centre-x, centre-y, w, h as fractions)."""
    img = Image.open(photo_path).convert('RGB')
    W, H = img.size
    lines = []
    for line in Path(box_path).read_text(errors='ignore').splitlines():
        parts = line.split()
        if not parts or parts[0].startswith('%'):
            continue
        name, l, t, w, h = parts[0], *map(float, parts[1:5])
        cls = CLASS_ID.get(name.lower())
        if cls is None:
            continue
        x1, y1 = max(l, 0), max(t, 0)
        x2, y2 = min(l + w, W), min(t + h, H)
        if x2 - x1 < 2 or y2 - y1 < 2:
            continue
        lines.append(f"{cls} {(x1+x2)/2/W:.6f} {(y1+y2)/2/H:.6f} {(x2-x1)/W:.6f} {(y2-y1)/H:.6f}")
    if max(W, H) > MAX_SIDE:
        img.thumbnail((MAX_SIDE, MAX_SIDE))
    out_photo.parent.mkdir(parents=True, exist_ok=True)
    out_label.parent.mkdir(parents=True, exist_ok=True)
    img.save(out_photo)
    out_label.write_text("\n".join(lines))

RAW = TEMP / 'data/raw'
for i, (key, photo) in enumerate(sorted(photos.items())):
    convert_one(photo, boxes[key], RAW/'images'/split_of[key]/f'{key}.png',
                RAW/'labels'/split_of[key]/f'{key}.txt')
    if i % 1500 == 0:
        print(f'{i}/{len(photos)}')
print('converted', len(photos), 'photos')

In [ ]:
import torch.nn as nn

class ZeroDCE(nn.Module):
    """Predicts 8 brightening curves; each is applied as  new = old + a*old*(1-old)."""
    def __init__(self, f=32):
        super().__init__()
        self.relu = nn.ReLU(inplace=True)
        self.e_conv1 = nn.Conv2d(3, f, 3, 1, 1);     self.e_conv2 = nn.Conv2d(f, f, 3, 1, 1)
        self.e_conv3 = nn.Conv2d(f, f, 3, 1, 1);     self.e_conv4 = nn.Conv2d(f, f, 3, 1, 1)
        self.e_conv5 = nn.Conv2d(f*2, f, 3, 1, 1);   self.e_conv6 = nn.Conv2d(f*2, f, 3, 1, 1)
        self.e_conv7 = nn.Conv2d(f*2, 24, 3, 1, 1)
    def forward(self, x):
        x1 = self.relu(self.e_conv1(x));  x2 = self.relu(self.e_conv2(x1))
        x3 = self.relu(self.e_conv3(x2)); x4 = self.relu(self.e_conv4(x3))
        x5 = self.relu(self.e_conv5(torch.cat([x3, x4], 1)))
        x6 = self.relu(self.e_conv6(torch.cat([x2, x5], 1)))
        for r in torch.split(torch.tanh(self.e_conv7(torch.cat([x1, x6], 1))), 3, dim=1):
            x = x + r * (x * x - x)
        return x

zerodce = ZeroDCE().cuda().eval()
zerodce.load_state_dict(torch.load(ZERODCE_FILE, map_location='cuda'))

@torch.no_grad()
def brighten(img):
    x = torch.from_numpy(img).permute(2, 0, 1).unsqueeze(0).float().cuda() / 255.0
    return (zerodce(x).clamp(0, 1)[0].permute(1, 2, 0).cpu().numpy() * 255).astype('uint8')

In [ ]:
def write_yaml(path, train_dir, val_dir, test_dir):
    """Tell YOLO where the photos are. Absolute paths, so 'path:' is just the filesystem root."""
    names = "\n".join(f'  {i}: {n}' for i, n in enumerate(CLASSES))
    Path(path).write_text(f"path: /\ntrain: {train_dir}\nval: {val_dir}\ntest: {test_dir}\nnames:\n{names}\n")
    return str(path)

def draw_boxes(img, boxes_xyxy, labels):
    out = img.copy()
    for (x1, y1, x2, y2), text in zip(boxes_xyxy, labels):
        cv2.rectangle(out, (int(x1), int(y1)), (int(x2), int(y2)), (0, 255, 0), 2)
        cv2.putText(out, text, (int(x1), max(int(y1)-5, 12)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
    return out

def find_in_inputs(filename):
    """Look for a file inside any attached Kaggle dataset / notebook output."""
    hits = sorted(Path('/kaggle/input').rglob(filename)) if Path('/kaggle/input').exists() else []
    return hits[0] if hits else None

## Step 2 · Brighten **all** photos, not just the test ones

E3 only needed brightened test photos. E4 trains on brightened photos, so the training and validation
sets must be brightened too — all 7,363. The labels are copied unchanged: brightening moves nothing.
(~10 min)

In [ ]:
ENH = TEMP / 'data/enhanced'
for split in ['train', 'val', 'test']:
    shutil.rmtree(ENH/'labels'/split, ignore_errors=True)
    shutil.copytree(RAW/'labels'/split, ENH/'labels'/split)
    (ENH/'images'/split).mkdir(parents=True, exist_ok=True)
    files = sorted((RAW/'images'/split).glob('*.png'))
    for i, p in enumerate(files):
        Image.fromarray(brighten(np.array(Image.open(p)))).save(ENH/'images'/split/p.name)
        if i % 1000 == 0:
            print(f'{split}: {i}/{len(files)}')
print('brightened every split')

# safety check: same file names, same sizes, identical labels in both versions
for split in ['train', 'val', 'test']:
    a = sorted(p.name for p in (RAW/'images'/split).glob('*.png'))
    b = sorted(p.name for p in (ENH/'images'/split).glob('*.png'))
    assert a == b, f'{split}: file lists differ'
    assert Image.open(RAW/'images'/split/a[0]).size == Image.open(ENH/'images'/split/a[0]).size
print('raw and brightened sets match, so the comparison is fair')

In [ ]:
# before / after, so you can show what the detector is now being trained on
demo_key = sorted(k for k in split_of if split_of[k] == 'train')[0]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].imshow(Image.open(RAW/'images/train'/f'{demo_key}.png')); axes[0].set_title('raw'); axes[0].axis('off')
axes[1].imshow(Image.open(ENH/'images/train'/f'{demo_key}.png')); axes[1].set_title('Zero-DCE'); axes[1].axis('off')
plt.show()

## Step 3 · Train `det_enh` on the brightened photos

**These settings are copied exactly from notebook 05.** Same model, same epochs, same batch size, same
seed — the *only* difference is which photos go in. If you change a setting here, E2 vs E4 no longer
measures brightening; it measures your setting change. (~45–75 min)

In [ ]:
from ultralytics import YOLO

yaml_enh_train = write_yaml(TEMP/'enh_train.yaml', ENH/'images/train', ENH/'images/val', ENH/'images/test')

model_enh = YOLO('yolov8s.pt')
model_enh.train(
    data=yaml_enh_train,
    epochs=60, imgsz=640, batch=32, device=DEVICE, seed=0,
    patience=15, save_period=5,
    project=str(WORK/'runs'), name='det_enh', exist_ok=True,
)
shutil.copy(WORK/'runs/det_enh/weights/best.pt', WORK/'det_enh.pt')
print('saved', WORK/'det_enh.pt')

## Step 4 · Score E4 — and E2/E3 too if `det_raw.pt` is attached

In [ ]:
yaml_raw_test = write_yaml(TEMP/'raw.yaml',  RAW/'images/train', RAW/'images/val', RAW/'images/test')
yaml_enh_test = write_yaml(TEMP/'enh.yaml',  RAW/'images/train', RAW/'images/val', ENH/'images/test')

results = {}
m_enh = YOLO(WORK/'det_enh.pt')
print('=== E4: trained on brightened, tested on brightened ===')
results['E4'] = m_enh.val(data=yaml_enh_test, split='test', imgsz=640, device=0)

det_raw_path = find_in_inputs('det_raw.pt')
if det_raw_path:
    print('\nfound', det_raw_path, '- also scoring E2 and E3')
    m_raw = YOLO(det_raw_path)
    results['E2'] = m_raw.val(data=yaml_raw_test, split='test', imgsz=640, device=0)
    results['E3'] = m_raw.val(data=yaml_enh_test, split='test', imgsz=640, device=0)
else:
    print('\nno det_raw.pt attached - E4 only. Compare with the E2 number from notebook 05.')

In [ ]:
LABEL = {'E2': 'E2  trained raw / tested raw',
         'E3': 'E3  trained raw / tested brightened',
         'E4': 'E4  trained brightened / tested brightened'}
order = [e for e in ['E2', 'E3', 'E4'] if e in results]

summary = pd.DataFrame(
    {LABEL[e]: [results[e].box.map50, results[e].box.map, results[e].box.mp, results[e].box.mr] for e in order},
    index=['mAP@0.5', 'mAP@0.5:0.95', 'precision', 'recall']).round(4)
print(summary.to_string())
summary.to_csv(WORK/'summary_E4.csv')

if 'E2' in results:
    d = results['E4'].box.map50 - results['E2'].box.map50
    print(f'\nMAIN RESULT  E4 - E2 = {d:+.4f} mAP@0.5')
    print('Brightening HELPED' if d > 0 else 'Brightening did NOT help')
    print('(a clear, well-explained negative result is a valid outcome for this project)')

In [ ]:
# per-class: where did brightening change things?
def ap_per_class(r):
    return [r.box.ap50[list(r.box.ap_class_index).index(i)] if i in r.box.ap_class_index else 0.0 for i in range(12)]

per_class = pd.DataFrame({e: ap_per_class(results[e]) for e in order}, index=CLASSES)
if 'E2' in results:
    per_class['E4 - E2'] = (per_class['E4'] - per_class['E2']).round(4)
print(per_class.round(4).to_string())
per_class.to_csv(WORK/'per_class_E4.csv')
per_class[order].plot.bar(figsize=(11, 4), title='AP@0.5 per class')
plt.ylabel('AP@0.5'); plt.tight_layout(); plt.savefig(WORK/'per_class_E4.png', dpi=150); plt.show()

### Does it help in the darkest photos?

In [ ]:
test_keys = sorted(k for k in split_of if split_of[k] == 'test')
rows = []
for light in ['Low', 'Weak', 'Ambient', 'Single', 'Strong', 'Twilight']:
    keys = [k for k in test_keys if lighting_of[k] == light]
    if len(keys) < 20:
        continue
    row = {'lighting': light, 'photos': len(keys)}
    for e in order:
        folder = RAW if e == 'E2' else ENH
        lst = TEMP/f'{e}_{light}.txt'
        lst.write_text("\n".join(str(folder/'images/test'/f'{k}.png') for k in keys))
        y = write_yaml(TEMP/f'y_{e}_{light}.yaml', RAW/'images/train', RAW/'images/val', lst)
        mdl = m_enh if e == 'E4' else m_raw
        row[e] = round(mdl.val(data=y, split='test', imgsz=640, device=0, verbose=False).box.map50, 4)
    rows.append(row)
by_light = pd.DataFrame(rows)
if 'E2' in order:
    by_light['E4 - E2'] = (by_light['E4'] - by_light['E2']).round(4)
print(by_light.to_string(index=False))
by_light.to_csv(WORK/'by_lighting_E4.csv', index=False)

## Step 5 · Pictures

In [ ]:
def detect_and_draw(model, image_path, conf=0.25):
    img = np.array(Image.open(image_path))
    r = model.predict(img[:, :, ::-1], conf=conf, device=0, verbose=False)[0]
    names = [f'{CLASSES[int(c)]} {s:.2f}' for c, s in zip(r.boxes.cls, r.boxes.conf)]
    return draw_boxes(img, r.boxes.xyxy.cpu().numpy(), names), len(names)

for key in test_keys[:4]:
    panels = []
    if 'E2' in results:
        panels.append(('E2 raw', *detect_and_draw(m_raw, RAW/'images/test'/f'{key}.png')))
    panels.append(('E4 brightened', *detect_and_draw(m_enh, ENH/'images/test'/f'{key}.png')))
    fig, axes = plt.subplots(1, len(panels), figsize=(7*len(panels), 5), squeeze=False)
    for ax, (title, img, n) in zip(axes[0], panels):
        ax.imshow(img); ax.set_title(f'{title} — {n} detections'); ax.axis('off')
    plt.tight_layout(); plt.savefig(WORK/f'example_E4_{key}.png', dpi=120); plt.show()

In [ ]:
print('Download these from the Output panel:')
for f in sorted(WORK.glob('*')):
    if f.is_file():
        print(f'  {f.name:28s} {f.stat().st_size/1e6:7.2f} MB')
print(f'\nTotal time: {(time.time()-START)/60:.0f} min')
print('''
Next (notebook 07): two follow-ups that explain the result
  E5  only brighten photos that are actually dark  (does over-brightening cause the damage?)
  E6  replace Zero-DCE with plain CLAHE            (is a neural network even needed?)
Attach this notebook's output so 07 can reuse det_enh.pt.
''')